# ROGII - Wellbore Geology Prediction
## Baseline: LightGBM + GR/Z-based TVT Prediction with Typewell Correlation

**Strategy:**
- For each horizontal well, the `TVT_input` column provides known TVT for the non-evaluation zone.
- For the evaluation zone (where `TVT_input` is NaN), we predict TVT using:
  1. **Per-well linear calibration**: fit a linear model Z → TVT on the known zone of each well, then extrapolate.
  2. **Cross-well LightGBM**: train a global model on features (Z, MD, X, Y, TVT_input_lag) from all training wells.
  3. **GR-Typewell sliding window correlation**: use GR log similarity to the typewell to refine the TVT position.
  4. **Blend** the above predictions.

**Key insight**: TVT is strongly correlated with Z (True Vertical Depth), and within a well's evaluation zone, TVT drifts only slightly. The per-well calibration captures well-specific geology.


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Data paths
DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

print('Data directory contents:')
print(os.listdir(DATA_DIR))

Data directory contents:
['sample_submission.csv', 'test', 'AI_wellbore_geology_prediction_task_en.pptx', 'train']


## 1. Load Training Data

In [2]:
def load_well_data(directory, well_name):
    """Load horizontal well and typewell CSV files for a given well name."""
    hw_path = os.path.join(directory, f'{well_name}__horizontal_well.csv')
    tw_path = os.path.join(directory, f'{well_name}__typewell.csv')
    hw = pd.read_csv(hw_path)
    tw = pd.read_csv(tw_path)
    return hw, tw


def get_well_names(directory):
    """Extract unique well names from a directory."""
    files = glob.glob(os.path.join(directory, '*__horizontal_well.csv'))
    return sorted([os.path.basename(f).replace('__horizontal_well.csv', '') for f in files])


train_wells = get_well_names(TRAIN_DIR)
test_wells = get_well_names(TEST_DIR)
print(f'Training wells: {len(train_wells)}')
print(f'Test wells: {len(test_wells)}')
print(f'Sample train wells: {train_wells[:5]}')

Training wells: 773
Test wells: 3
Sample train wells: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


In [3]:
def engineer_features(hw_df, tw_df):
    """
    Engineer features for the horizontal well dataframe using typewell information.
    
    Features created:
    - neg_Z: negative of Z (highly correlated with TVT)
    - TVT_input_filled: forward-fill of TVT_input (last known TVT)
    - TVT_offset: residual between neg_Z and TVT_input_filled
    - GR_filled: interpolated GR log
    - GR_tw_at_tvt: typewell GR looked up at the TVT_input_filled position
    - GR_diff: difference between horizontal well GR and typewell GR at that TVT
    - MD_since_eval_start: MD measured from the start of the evaluation zone
    """
    df = hw_df.copy()
    
    # Negative Z is ~TVT + constant
    df['neg_Z'] = -df['Z']
    
    # Forward-fill TVT_input: last known TVT before the eval zone
    df['TVT_input_filled'] = df['TVT_input'].ffill()
    df['TVT_input_filled'] = df['TVT_input_filled'].fillna(df['neg_Z'])
    
    # The deviation between neg_Z and last known TVT (drift proxy)
    df['TVT_offset_from_negZ'] = df['TVT_input_filled'] - df['neg_Z']
    
    # Distance in MD from the start of the eval zone
    eval_start_md = df.loc[df['TVT_input'].isna(), 'MD'].min()
    if pd.isna(eval_start_md):
        eval_start_md = df['MD'].max()
    df['MD_since_eval_start'] = (df['MD'] - eval_start_md).clip(lower=0)
    
    # Interpolate GR (sparse/NaN in some rows)
    df['GR_filled'] = df['GR'].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
    df['GR_filled'] = df['GR_filled'].fillna(df['GR_filled'].median())
    
    # GR rolling statistics
    df['GR_roll_mean_10'] = df['GR_filled'].rolling(10, min_periods=1).mean()
    df['GR_roll_std_10'] = df['GR_filled'].rolling(10, min_periods=1).std().fillna(0)
    df['GR_roll_mean_50'] = df['GR_filled'].rolling(50, min_periods=1).mean()
    
    # Look up typewell GR at the TVT_input_filled position (nearest-neighbor interpolation)
    tw_sorted = tw_df.sort_values('TVT').reset_index(drop=True)
    df['GR_tw_at_tvt'] = np.interp(
        df['TVT_input_filled'].values,
        tw_sorted['TVT'].values,
        tw_sorted['GR'].values
    )
    df['GR_diff'] = df['GR_filled'] - df['GR_tw_at_tvt']
    
    # Lag features for TVT_input (how fast TVT was changing before eval zone)
    df['TVT_input_diff'] = df['TVT_input'].diff().fillna(0)
    df['TVT_input_diff_filled'] = df['TVT_input_diff'].fillna(0)
    
    return df


FEATURE_COLS = [
    'MD', 'X', 'Y', 'Z', 'neg_Z',
    'TVT_input_filled', 'TVT_offset_from_negZ',
    'MD_since_eval_start',
    'GR_filled', 'GR_roll_mean_10', 'GR_roll_std_10', 'GR_roll_mean_50',
    'GR_tw_at_tvt', 'GR_diff',
    'TVT_input_diff_filled'
]

print('Feature columns:', FEATURE_COLS)

Feature columns: ['MD', 'X', 'Y', 'Z', 'neg_Z', 'TVT_input_filled', 'TVT_offset_from_negZ', 'MD_since_eval_start', 'GR_filled', 'GR_roll_mean_10', 'GR_roll_std_10', 'GR_roll_mean_50', 'GR_tw_at_tvt', 'GR_diff', 'TVT_input_diff_filled']


In [4]:
# Load all training data and build the global training dataset
all_train_rows = []
print('Loading training wells...')

for i, well in enumerate(train_wells):
    hw, tw = load_well_data(TRAIN_DIR, well)
    hw = engineer_features(hw, tw)
    hw['well_name'] = well
    
    # Keep only rows where TVT is known (training targets)
    # TVT_input is NaN in the eval zone, but TVT itself is provided in training
    hw_known = hw[hw['TVT_input'].notna()].copy()
    all_train_rows.append(hw_known)
    
    if (i + 1) % 20 == 0:
        print(f'  Loaded {i+1}/{len(train_wells)} wells')

train_df = pd.concat(all_train_rows, ignore_index=True)
print(f'\nTotal training rows (known zone only): {len(train_df)}')
print(f'Features: {train_df[FEATURE_COLS].shape}')
print(train_df[FEATURE_COLS].describe())

Loading training wells...
  Loaded 20/773 wells
  Loaded 40/773 wells
  Loaded 60/773 wells
  Loaded 80/773 wells
  Loaded 100/773 wells
  Loaded 120/773 wells
  Loaded 140/773 wells
  Loaded 160/773 wells
  Loaded 180/773 wells
  Loaded 200/773 wells
  Loaded 220/773 wells
  Loaded 240/773 wells
  Loaded 260/773 wells
  Loaded 280/773 wells
  Loaded 300/773 wells
  Loaded 320/773 wells
  Loaded 340/773 wells
  Loaded 360/773 wells
  Loaded 380/773 wells
  Loaded 400/773 wells
  Loaded 420/773 wells
  Loaded 440/773 wells
  Loaded 460/773 wells
  Loaded 480/773 wells
  Loaded 500/773 wells
  Loaded 520/773 wells
  Loaded 540/773 wells
  Loaded 560/773 wells
  Loaded 580/773 wells
  Loaded 600/773 wells
  Loaded 620/773 wells
  Loaded 640/773 wells
  Loaded 660/773 wells
  Loaded 680/773 wells
  Loaded 700/773 wells
  Loaded 720/773 wells
  Loaded 740/773 wells
  Loaded 760/773 wells

Total training rows (known zone only): 1308266
Features: (1308266, 15)
                 MD             

## 2. Train LightGBM Model

In [5]:
# Prepare training arrays
X_train = train_df[FEATURE_COLS].values
y_train = train_df['TVT'].values

print(f'Training set: X={X_train.shape}, y={y_train.shape}')

# LightGBM model parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

model = lgb.LGBMRegressor(**lgb_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    callbacks=[lgb.log_evaluation(100)]
)

train_pred = model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f'\nTrain RMSE (known zone): {train_rmse:.4f}')

Training set: X=(1308266, 15), y=(1308266,)
[100]	training's rmse: 7.41558
[200]	training's rmse: 4.77771
[300]	training's rmse: 4.28598
[400]	training's rmse: 3.98839
[500]	training's rmse: 3.78574
[600]	training's rmse: 3.62084
[700]	training's rmse: 3.49425
[800]	training's rmse: 3.38578
[900]	training's rmse: 3.28978
[1000]	training's rmse: 3.20763

Train RMSE (known zone): 3.2076


In [6]:
# Feature importance
feat_imp = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print('Feature importances:')
print(feat_imp.to_string(index=False))

Feature importances:
              feature  importance
     TVT_input_filled        9905
 TVT_offset_from_negZ        9549
                    Z        6238
                   MD        5331
      GR_roll_mean_50        4616
                    Y        4398
                neg_Z        4377
TVT_input_diff_filled        4373
                    X        4098
         GR_tw_at_tvt        2852
      GR_roll_mean_10        2518
       GR_roll_std_10        2052
            GR_filled         901
              GR_diff         792
  MD_since_eval_start           0


## 3. Per-Well Linear Calibration

For each test well, we fit a local linear regression Z → TVT on the known zone. This handles well-specific geological depth offsets that the global model may not capture perfectly.

In [7]:
def fit_per_well_linear(hw_known_df):
    """
    Fit a per-well linear model: TVT = a*Z + b*MD + c
    Returns the fitted LinearRegression object.
    """
    X = hw_known_df[['Z', 'MD']].values
    y = hw_known_df['TVT_input'].values
    lr = LinearRegression()
    lr.fit(X, y)
    return lr


def predict_with_blend(hw_full_df, tw_df, global_model, blend_alpha=0.7):
    """
    Predict TVT for the evaluation zone of a well.
    
    Blends:
    - global LightGBM prediction (weight: blend_alpha)
    - per-well linear regression prediction (weight: 1 - blend_alpha)
    
    Args:
        hw_full_df: Full horizontal well dataframe (with engineered features)
        tw_df: Typewell dataframe
        global_model: Trained LightGBM model
        blend_alpha: Weight for the global model (0 = linear only, 1 = LightGBM only)
    
    Returns:
        DataFrame with row index and predicted TVT
    """
    known_mask = hw_full_df['TVT_input'].notna()
    eval_mask = hw_full_df['TVT_input'].isna()
    
    hw_known = hw_full_df[known_mask]
    hw_eval = hw_full_df[eval_mask]
    
    if len(hw_eval) == 0:
        return pd.DataFrame(columns=['row_idx', 'tvt_pred'])
    
    # Global LightGBM prediction
    X_eval = hw_eval[FEATURE_COLS].values
    lgb_pred = global_model.predict(X_eval)
    
    # Per-well linear prediction
    if len(hw_known) >= 2:
        lr = fit_per_well_linear(hw_known)
        lin_pred = lr.predict(hw_eval[['Z', 'MD']].values)
    else:
        # Fallback: use last known TVT
        last_tvt = hw_known['TVT_input'].iloc[-1] if len(hw_known) > 0 else 0
        lin_pred = np.full(len(hw_eval), last_tvt)
    
    # Blend predictions
    blended = blend_alpha * lgb_pred + (1 - blend_alpha) * lin_pred
    
    return pd.DataFrame({
        'row_idx': hw_eval.index.tolist(),
        'tvt_pred': blended
    })


print('Per-well prediction functions defined.')

Per-well prediction functions defined.


## 4. Validate on Training Wells (Cross-validation on eval zones)

In [8]:
# Validate on a subset of training wells that have eval zones
# Since we trained only on the known zone, we can evaluate on the eval zone (where TVT is known in training)

val_wells = train_wells[:20]  # Use first 20 wells for quick validation
val_results = []

for well in val_wells:
    hw, tw = load_well_data(TRAIN_DIR, well)
    hw = engineer_features(hw, tw)
    
    eval_mask = hw['TVT_input'].isna()
    if eval_mask.sum() == 0:
        continue
    
    preds = predict_with_blend(hw, tw, model, blend_alpha=0.7)
    true_tvt = hw.loc[eval_mask, 'TVT'].values
    
    rmse = np.sqrt(mean_squared_error(true_tvt, preds['tvt_pred'].values))
    val_results.append({'well': well, 'rmse': rmse, 'n_rows': len(true_tvt)})

val_df = pd.DataFrame(val_results)
print('Validation on training eval zones:')
print(val_df.to_string(index=False))

# Weighted average RMSE
total_rows = val_df['n_rows'].sum()
weighted_rmse = np.sqrt((val_df['rmse']**2 * val_df['n_rows']).sum() / total_rows)
print(f'\nWeighted validation RMSE: {weighted_rmse:.4f}')

Validation on training eval zones:
    well      rmse  n_rows
000d7d20 10.750203    3836
00bbac68 29.322546    6014
00e12e8b 14.676974    4301
015fe0d2  6.455870    4296
01869cd4 19.108783    5557
01982c1d  6.337020    3941
028d7b28 16.933170    6234
02e7fe5a  7.144753    5108
0390d174 10.919937    4411
03a935ae 13.448082    2786
044af7d1 10.674278    4519
0498acab  9.557364    4798
052d64df 12.499002    2445
05948241  4.786579    3879
059c8f24 17.234846    6258
05a0ee4d 31.941916    4348
060ab2b8 33.630810    6258
06df5958 11.016300    3509
071d7b45  6.731736    3211
0849b4e2  7.612357    5537

Weighted validation RMSE: 17.4154


## 5. Generate Test Predictions

In [9]:
# Load the sample submission to get all required IDs
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f'Sample submission rows: {len(sample_sub)}')

# Parse well name and row index from ID
sample_sub['well_name'] = sample_sub['id'].str[:8]
sample_sub['row_idx'] = sample_sub['id'].str.split('_').str[-1].astype(int)
print(f'Test wells in submission: {sample_sub["well_name"].nunique()}')
print(sample_sub.head())

Sample submission rows: 14151
Test wells in submission: 3
              id  tvt well_name  row_idx
0  000d7d20_1442  0.0  000d7d20     1442
1  000d7d20_1443  0.0  000d7d20     1443
2  000d7d20_1444  0.0  000d7d20     1444
3  000d7d20_1445  0.0  000d7d20     1445
4  000d7d20_1446  0.0  000d7d20     1446


In [10]:
# Generate predictions for all test wells
all_predictions = []

print('Generating predictions for test wells...')
for i, well in enumerate(test_wells):
    # Load test well data
    hw, tw = load_well_data(TEST_DIR, well)
    hw = engineer_features(hw, tw)
    
    # Predict eval zone
    preds = predict_with_blend(hw, tw, model, blend_alpha=0.7)
    preds['well_name'] = well
    all_predictions.append(preds)
    
    if (i + 1) % 20 == 0:
        print(f'  Processed {i+1}/{len(test_wells)} wells')

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f'\nTotal prediction rows: {len(pred_df)}')

Generating predictions for test wells...

Total prediction rows: 14151


In [11]:
# Merge predictions with submission format
# The submission ID format is: {wellname}_{row_index}
# row_index is the 0-based integer index in the original horizontal_well CSV

pred_df['id'] = pred_df['well_name'] + '_' + pred_df['row_idx'].astype(str)

submission = sample_sub[['id']].merge(
    pred_df[['id', 'tvt_pred']],
    on='id',
    how='left'
)

# Fill any missing predictions with the median (safety fallback)
median_tvt = pred_df['tvt_pred'].median()
missing_count = submission['tvt_pred'].isna().sum()
if missing_count > 0:
    print(f'Warning: {missing_count} missing predictions, filling with median {median_tvt:.2f}')
    submission['tvt_pred'] = submission['tvt_pred'].fillna(median_tvt)

submission = submission.rename(columns={'tvt_pred': 'tvt'})

print(f'Submission rows: {len(submission)}')
print(f'Missing values: {submission["tvt"].isna().sum()}')
print(submission.head(10))
print(f'\nTVT prediction stats:')
print(submission['tvt'].describe())

Submission rows: 14151
Missing values: 0
              id           tvt
0  000d7d20_1442  11752.173171
1  000d7d20_1443  11752.027062
2  000d7d20_1444  11752.029899
3  000d7d20_1445  11752.181683
4  000d7d20_1446  11752.184520
5  000d7d20_1447  11752.187358
6  000d7d20_1448  11752.187215
7  000d7d20_1449  11752.190053
8  000d7d20_1450  11752.192890
9  000d7d20_1451  11752.195727

TVT prediction stats:
count    14151.000000
mean     11898.582793
std        259.523113
min      11605.320614
25%      11618.666136
50%      11755.517543
75%      12190.203950
max      12222.534289
Name: tvt, dtype: float64


In [12]:
# Save submission
submission[['id', 'tvt']].to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission[['id', 'tvt']].head(10).to_string(index=False))

submission.csv saved!
           id          tvt
000d7d20_1442 11752.173171
000d7d20_1443 11752.027062
000d7d20_1444 11752.029899
000d7d20_1445 11752.181683
000d7d20_1446 11752.184520
000d7d20_1447 11752.187358
000d7d20_1448 11752.187215
000d7d20_1449 11752.190053
000d7d20_1450 11752.192890
000d7d20_1451 11752.195727


## 6. Summary

**Model:** LightGBM + per-well Linear Regression blend

**Key features:**
- `neg_Z` (−Z, True Vertical Depth negated) — highest correlation with TVT
- `TVT_input_filled` — last known TVT (forward-filled)
- `TVT_offset_from_negZ` — drift between −Z and last known TVT
- `MD`, `X`, `Y` — wellbore spatial position
- `GR_filled`, `GR_roll_mean_10/50`, `GR_roll_std_10` — gamma ray log and rolling statistics
- `GR_tw_at_tvt`, `GR_diff` — typewell GR at the current TVT position and the difference
- `MD_since_eval_start` — distance into the evaluation zone

**Blend strategy:**
- 70% global LightGBM (captures cross-well patterns)
- 30% per-well linear Z+MD regression (captures well-specific depth offset)

**Next steps to improve:**
- Dynamic Time Warping (DTW) between horizontal GR and typewell GR
- Sequence models (LSTM/Transformer) for smooth TVT trajectory prediction
- Better handling of GR NaN values using surrounding context
- Tuning the blend ratio per well based on local confidence